[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/06_%E5%85%B1%E5%88%86%E6%95%A3%E3%83%BB%E5%A4%89%E5%8B%95%E4%BF%82%E6%95%B0%E3%83%BB%E6%AD%A3%E8%A6%8F%E8%BF%91%E4%BC%BC.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計3級-06. 共分散・変動係数・二項分布の正規近似

**🎯 できるようになること**
- 共分散から相関係数を導ける
- 変動係数でスケールの違うばらつきを比べられる
- 二項分布の正規近似を使える

**前提**：統計3級01〜03　/　**所要**：約30分　/　**レベル**：統計検定3級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む

### 📋 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

## 1. 共分散

2つの量が「一緒に大きくなる／小さくなる」傾向を表す数。
$$ s_{xy} = \frac{1}{n}\sum (x_i - \bar{x})(y_i - \bar{y}) $$

正なら正の関係、負なら負の関係。ただし**単位に依存する**のが弱点。

In [ ]:
x = df['数学'].values                      # 数学（配列）
y = df['英語'].values                      # 英語（配列）
cov = np.mean((x - x.mean()) * (y - y.mean()))   # 共分散＝偏差の積の平均（÷n）
print('共分散:', round(cov, 2))
print('np.cov(÷n-1):', round(np.cov(x, y, ddof=0)[0,1], 2))  # numpyでも確認（ddof=0で÷n）

## 2. 共分散 → 相関係数

共分散を各標準偏差で割ると、単位によらない −1〜1 の**相関係数**になります。
$$ r = \frac{s_{xy}}{s_x \, s_y} $$

In [ ]:
r = cov / (x.std() * y.std())              # 相関係数＝共分散÷(各標準偏差)
print('相関係数(共分散から):', round(r, 3))
print('pandas .corr() で確認:', round(df['数学'].corr(df['英語']), 3))

## 3. 変動係数（CV）

標準偏差を平均で割ったもの。**単位やスケールが違うデータのばらつきを比べる**のに使う。
$$ CV = \frac{標準偏差}{平均} $$

例：身長(cm)と体重(kg)、どちらが相対的にばらつくか。

In [ ]:
height = np.array([158, 162, 170, 165, 168, 160])  # 身長
weight = np.array([48, 55, 70, 60, 65, 50])        # 体重
# それぞれ 変動係数＝標準偏差÷平均 を計算
for name, d in [('身長', height), ('体重', weight)]:
    cv = d.std() / d.mean()                # 変動係数
    print(f'{name}: 標準偏差{d.std():.1f}, 平均{d.mean():.1f}, CV={cv:.3f}')
print('→ CVが大きい方が、平均に対して相対的にばらついている')

## 4. 二項分布の正規近似

二項分布 $B(n, p)$ は、$n$ が大きいと正規分布 $N(np,\ np(1-p))$ に近づきます。
コインを多数回投げたときの表の回数で確かめます。

In [ ]:
n, p = 100, 0.5                            # 試行回数と成功確率
k = np.arange(0, n + 1)                    # 成功回数の候補
binom = stats.binom.pmf(k, n, p)           # 二項分布の確率
mu, sigma = n * p, np.sqrt(n * p * (1 - p))  # 対応する正規分布の平均と標準偏差
xs = np.linspace(0, n, 300)                # 正規曲線用のx
normal = stats.norm.pdf(xs, mu, sigma)     # 正規分布の高さ

plt.bar(k, binom, alpha=0.5, label='二項分布 B(100,0.5)')   # 二項分布（棒）
plt.plot(xs, normal, 'r-', label=f'正規近似 N({mu:.0f},{sigma**2:.0f})')  # 正規近似（曲線）
plt.legend(); plt.title('二項分布の正規近似'); plt.show()
print(f'平均 np={mu}, 標準偏差 √(np(1-p))={sigma:.2f}')

### 近似を使った確率計算
「表が60回以上」の確率を、二項分布そのものと正規近似で比べます。

In [ ]:
exact = 1 - stats.binom.cdf(59, n, p)      # 二項分布での厳密な P(X≥60)
# 連続補正（59.5を境にする）を入れた正規近似
approx = 1 - stats.norm.cdf(59.5, mu, sigma)  # 正規近似での P(X≥60)
print(f'二項分布での厳密値: {exact:.4f}')
print(f'正規近似(連続補正): {approx:.4f}  → ほぼ一致')

> ⚠️ **よくある間違い**：共分散は**単位に依存**するので、値の大小だけで関係の強さは比べられない。強さを比べるには無単位の**相関係数**（共分散÷各標準偏差）にする。

## 5. 単位を変えると…共分散は変わる、相関は変わらない

変数を一次変換（`a倍して b を足す`）すると、**共分散は倍率の積だけ変化**しますが、**相関係数は倍率では変わらず、符号だけが傾きの符号で決まります**。

In [ ]:
rng = np.random.default_rng(1)
X = rng.normal(50, 10, 200)
Y = 0.7*X + rng.normal(0, 8, 200)
cov = np.cov(X, Y, ddof=0)[0,1]; r = np.corrcoef(X, Y)[0,1]
U = 2*X + 3; V = -5*Y + 1                 # X,Y をそれぞれ一次変換
print(f'Cov[X,Y] = {cov:.1f}   r[X,Y] = {r:.3f}')
print(f'Cov[U,V] = {np.cov(U,V,ddof=0)[0,1]:.1f}  ≈ 2×(-5)×Cov = {2*-5*cov:.1f}')
print(f'r[U,V]   = {np.corrcoef(U,V)[0,1]:.3f}  ≈ sign(2×-5)×r = {-r:.3f}')

> 💡 共分散は単位（倍率）で値が変わるので、大小だけで関係の強さは比べられません。だから単位によらない **相関係数** で「関係の強さ」を見ます。一次変換では符号だけが反転し得ます。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import numpy as np
x = [1,2,3]; y = [2,4,6]
# Q1: x,y の共分散（÷n）を ans に
ans = None   # 例: np.cov(x, y, ddof=0)[0,1]
_check('Q1 共分散', ans, np.cov(x,y,ddof=0)[0,1])

In [ ]:
import numpy as np
d = [10, 20, 30]
# Q2: d の変動係数（標準偏差÷平均, ÷n）を ans に
ans = None   # 例: np.std(d)/np.mean(d)
_check('Q2 変動係数', ans, np.std(d)/np.mean(d))

In [ ]:
import numpy as np
# Q3: 二項分布 B(100, 0.5) の標準偏差 √(np(1-p)) を ans に
ans = None   # 例: np.sqrt(100*0.5*0.5)
_check('Q3 標準偏差', ans, np.sqrt(100*0.5*0.5))

In [ ]:
import numpy as np
x = np.array([1,2,3,4,5]); y = np.array([2,4,5,4,6])
# Q4: x を「×10」, y を「×(-1)」したときの相関係数を ans に（符号が反転するはず）
ans = None   # 例: np.corrcoef(x*10, y*(-1))[0,1]
_check('Q4 変換後の相関', ans, np.corrcoef(x*10, y*(-1))[0,1])

---
## 🏆 練習問題

**問1.** `df` の `勉強時間` と `数学` の共分散を手計算の式で求め、
それを標準偏差で割って相関係数になることを確かめよう。

**問2.** A店の日販 `[50,52,48,51,49]`（万円）、B店 `[10,14,8,12,16]`（万円）。
変動係数を比べ、どちらが相対的に安定しているか答えよう。

**問3.** サイコロを180回ふるとき「1の目」が出る回数は二項分布 B(180, 1/6)。
平均と標準偏差を求め、正規近似で「35回以上出る確率」を計算しよう。

**問4.** `students_scores.csv` の数学を「×100」しても、英語との相関係数が変わらないことを確かめよう。次に英語を「×(-1)」したら相関の符号はどうなる？

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[06_共分散・変動係数・正規近似 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/06_%E5%85%B1%E5%88%86%E6%95%A3%E3%83%BB%E5%A4%89%E5%8B%95%E4%BF%82%E6%95%B0%E3%83%BB%E6%AD%A3%E8%A6%8F%E8%BF%91%E4%BC%BC.md)**

🎉 これで**3級の出題範囲をほぼ全てカバー**しました。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 共分散 | 一緒に増減する傾向（単位依存） |
| 相関係数 | 共分散÷(各標準偏差)、−1〜1 |
| 変動係数 | 標準偏差÷平均（相対ばらつき） |
| 正規近似 | nが大きい二項分布≈正規分布 |
| 連続補正 | 離散→連続の0.5のずらし |

In [ ]:
# チートシート（共分散・近似）
import numpy as np
np.cov(x, y, ddof=0)[0,1]            # 共分散(÷n)
df['数学'].corr(df['英語'])           # 相関係数
df['数学'].std()/df['数学'].mean()    # 変動係数
np.sqrt(n*p*(1-p))                   # 二項の標準偏差